# Cost-Optimal Data Center Self-Supply Under Curtailment

This notebook builds and solves the two-stage capacity-expansion LP in `draft_model.tex`: first-stage power and energy investments, followed by expected event dispatch and load shifting.

**Scenario structure:** the default design has 84 equally weighted events from 3 start hours (16, 17, 18), 7 durations (3–9 hours), and 4 curtailment depths (0.25, 0.50, 0.75, 1.0). Scenario-specific demand and resource-availability profiles must be prepared and supplied; this notebook does not fetch input data.

**Implementation notes:** energy capacity and state of charge are modeled only for candidates in `G^E = G^DIES ∪ G^BESS`. Each event starts with full fuel/charge and is independent of other events. The load-shift decision `sigma` is a fraction of baseline demand, matching `(kappa - sigma) * D`. The discharge state equation uses the physical sign: `b[t+1] = b[t] - theta[t] / eta`.

**Prerequisites:** the active Julia environment must have `JuMP` and `Gurobi` installed (`import Pkg; Pkg.add(["JuMP", "Gurobi"])` if needed).

In [35]:
using JuMP
using Gurobi
using Printf

## Input data contract

Each scenario index represents one event assembled from a start hour, duration, and curtailment depth. The draft's default scenario grid has 3 start hours (16, 17, 18), 7 durations (3–9 hours), and 4 depths (0.25, 0.50, 0.75, 1.0): 84 equally weighted scenarios. Provide the scenario-specific load and availability arrays after combining these definitions with representative load/resource profiles. No input data are fetched here.

All capacities and profiles use consistent MW, MWh, and currency units. Padded profile columns beyond each scenario's duration are ignored.

| Field | Shape | Meaning |
|---|---:|---|
| `p_max`, `p_cap_cost`, `resource_type` | generators | Maximum buildable MW, annualized capacity cost per MW, and type label (e.g. `BESS`, `Diesel`, `VPP`, `PV`) |
| `energy_indices` | vector | Indices of `G^E = G^DIES ∪ G^BESS` among all generators; MWh is aggregated by their `resource_type` |
| `e_max`, `e_cap_cost`, `eta` | generators | Maximum MWh, annualized energy cost per MWh, and efficiency (used only for `energy_indices`) |
| `marginal_cost`, `emissions_rate` | generators | Dispatch cost and emissions per MWh |
| `availability` | scenarios × generators × padded hours | `alpha[omega,g,t]` availability fraction |
| `scenario_weight`, `duration`, `kappa`, `start_hour` | scenarios | Scenario probability, event length, constant event curtailment depth, and start hour metadata |
| `demand` | scenarios × padded hours | Baseline data-center load profile `D[omega,t]` |
| `sigma_max` | scalar | Maximum load-shift fraction, as used in `(kappa - sigma) * demand` |
| `events_per_year` | scalar | Annual event count `N` (19 base case; 6 and 38 sensitivities) |
| `carbon_price`, `unserved_penalty` | scalars | Carbon price and penalty per unserved MWh |

`scenario_weight` must sum to 1 (the specified 84-scenario case uses `1/84` per scenario). `sigma` is an optimization variable bounded between 0 and `sigma_max`. Power build cost is charged once; expected event operating costs are multiplied by `events_per_year`.

Diagnostics report cost components and aggregate build by resource type. The balance dual is reported for displayed scenario-hours; it is the JuMP dual for the equality written as `dispatch + unserved = curtailed load remaining`, with the scenario probability and annual event-count scaling already present in the objective. Only a few representative scenario dispatch tables are printed when the scenario set is large.

In [36]:
function validate_data(data)
    required = (
        :p_max, :p_cap_cost, :energy_indices, :e_max, :e_cap_cost, :eta,
        :marginal_cost, :emissions_rate, :resource_type, :availability, :scenario_weight,
        :duration, :kappa, :start_hour, :demand, :sigma_max,
        :events_per_year, :carbon_price, :unserved_penalty,
    )
    for field in required
        hasproperty(data, field) || throw(ArgumentError("missing data field: $field"))
    end

    ng = length(data.p_max)
    nω = length(data.duration)
    nt = size(data.demand, 2)
    energy_indices = data.energy_indices
    ng > 0 || throw(ArgumentError("at least one generator candidate is required"))
    nω > 0 || throw(ArgumentError("at least one scenario is required"))
    nt > 0 || throw(ArgumentError("at least one padded time period is required"))
    length(unique(energy_indices)) == length(energy_indices) ||
        throw(ArgumentError("energy_indices must not contain duplicates"))
    all(g -> 1 <= g <= ng, energy_indices) ||
        throw(ArgumentError("energy_indices must refer to existing generators"))

    length(data.resource_type) == ng ||
        throw(ArgumentError("resource_type must have one label per generator"))
    for field in (:p_cap_cost, :e_max, :e_cap_cost, :eta, :marginal_cost, :emissions_rate)
        length(getproperty(data, field)) == ng ||
            throw(ArgumentError("$field must have one value per generator"))
    end
    for field in (:scenario_weight, :kappa, :start_hour)
        length(getproperty(data, field)) == nω ||
            throw(ArgumentError("$field must have one value per scenario"))
    end
    size(data.demand, 1) == nω ||
        throw(ArgumentError("demand must have one row per scenario"))
    size(data.availability) == (nω, ng, nt) ||
        throw(ArgumentError("availability must have shape (scenarios, generators, padded periods)"))
    all(t -> t isa Integer && 1 <= t <= nt, data.duration) ||
        throw(ArgumentError("each scenario duration must be an integer in 1:padded_horizon"))

    for field in (:p_max, :p_cap_cost, :marginal_cost, :emissions_rate, :scenario_weight)
        all(x -> isfinite(x) && x >= 0, getproperty(data, field)) ||
            throw(ArgumentError("$field values must be finite and nonnegative"))
    end
    all(g -> isfinite(data.e_max[g]) && data.e_max[g] >= 0, energy_indices) ||
        throw(ArgumentError("e_max values for energy-limited generators must be finite and nonnegative"))
    all(g -> isfinite(data.e_cap_cost[g]) && data.e_cap_cost[g] >= 0, energy_indices) ||
        throw(ArgumentError("e_cap_cost values for energy-limited generators must be finite and nonnegative"))
    all(g -> isfinite(data.eta[g]) && 0 < data.eta[g] <= 1, energy_indices) ||
        throw(ArgumentError("eta values for energy-limited generators must be in (0, 1]"))
    isapprox(sum(data.scenario_weight), 1.0; atol = 1e-8) ||
        throw(ArgumentError("scenario_weight values must sum to 1"))
    all(x -> isfinite(x) && 0 <= x <= 1, data.kappa) ||
        throw(ArgumentError("kappa values must be in [0, 1]"))
    all(x -> isfinite(x) && x >= 0, data.demand) ||
        throw(ArgumentError("demand values must be finite and nonnegative"))
    all(x -> isfinite(x) && 0 <= x <= 1, data.availability) ||
        throw(ArgumentError("availability values must be in [0, 1]"))
    isfinite(data.sigma_max) && data.sigma_max >= 0 ||
        throw(ArgumentError("sigma_max must be finite and nonnegative"))
    isfinite(data.events_per_year) && data.events_per_year >= 0 ||
        throw(ArgumentError("events_per_year must be finite and nonnegative"))
    isfinite(data.carbon_price) && data.carbon_price >= 0 ||
        throw(ArgumentError("carbon_price must be finite and nonnegative"))
    isfinite(data.unserved_penalty) && data.unserved_penalty >= 0 ||
        throw(ArgumentError("unserved_penalty must be finite and nonnegative"))
    return nothing
end

function build_model(data; optimizer = Gurobi.Optimizer)
    validate_data(data)
    ng = length(data.p_max)
    nω = length(data.duration)
    nt = size(data.demand, 2)
    energy_indices = data.energy_indices

    model = Model(optimizer)
    set_silent(model)

    @variable(model, P[1:ng] >= 0)
    @variable(model, E[g in energy_indices] >= 0)
    @variable(model, theta[1:ng, 1:nω, 1:nt] >= 0)
    @variable(model, bE[g in energy_indices, ω in 1:nω, t in 1:(nt + 1)] >= 0)
    @variable(model, 0 <= sigma[1:nω, 1:nt] <= data.sigma_max)
    @variable(model, u[1:nω, 1:nt] >= 0)

    @constraint(model, [g in 1:ng], P[g] <= data.p_max[g])
    @constraint(model, [g in energy_indices], E[g] <= data.e_max[g])

    balance = Dict{Tuple{Int,Int},ConstraintRef}()
    for ω in 1:nω
        for g in 1:ng
            for t in 1:nt
                @constraint(model, theta[g, ω, t] <= data.availability[ω, g, t] * P[g])
            end
        end
        for g in energy_indices
            @constraint(model, bE[g, ω, 1] == E[g])
            for t in 1:(nt + 1)
                @constraint(model, bE[g, ω, t] <= E[g])
            end
            for t in 1:data.duration[ω]
                @constraint(model,
                    bE[g, ω, t + 1] == bE[g, ω, t] - theta[g, ω, t] / data.eta[g])
            end
        end
        for t in 1:data.duration[ω]
            balance[(ω, t)] = @constraint(model,
                sum(theta[g, ω, t] for g in 1:ng) + u[ω, t] ==
                (data.kappa[ω] - sigma[ω, t]) * data.demand[ω, t])
        end
    end

    @objective(model, Min,
        sum(data.p_cap_cost[g] * P[g] for g in 1:ng) +
        sum(data.e_cap_cost[g] * E[g] for g in energy_indices) +
        data.events_per_year * sum(data.scenario_weight[ω] * (
                sum((data.marginal_cost[g] + data.carbon_price * data.emissions_rate[g]) *
                    theta[g, ω, t] for g in 1:ng) +
                data.unserved_penalty * u[ω, t])
            for ω in 1:nω for t in 1:data.duration[ω]))

    return model, (P = P, E = E, theta = theta, sigma = sigma, u = u, bE = bE, balance = balance)
end

function solve_model(data; optimizer = Gurobi.Optimizer)
    model, variables = build_model(data; optimizer)
    optimize!(model)
    has_values(model) || error("No feasible solution returned (status: $(termination_status(model)))")

    power_capacity = value.(variables.P)
    energy_capacity = [value(variables.E[g]) for g in data.energy_indices]
    dispatch = value.(variables.theta)
    unserved = value.(variables.u)
    energy_state = Array{Float64}(undef,
        length(data.energy_indices), length(data.duration), size(data.demand, 2) + 1)
    for (i, g) in enumerate(data.energy_indices), ω in eachindex(data.duration),
        t in 1:(size(data.demand, 2) + 1)
        energy_state[i, ω, t] = value(variables.bE[g, ω, t])
    end

    nω, nt = size(data.demand)
    balance_dual = fill(NaN, nω, nt)
    scenario_operating_cost = zeros(nω)
    for ω in 1:nω, t in 1:data.duration[ω]
        balance_dual[ω, t] = dual(variables.balance[(ω, t)])
        scenario_operating_cost[ω] += data.events_per_year * data.scenario_weight[ω] *
            (sum((data.marginal_cost[g] + data.carbon_price * data.emissions_rate[g]) *
                 dispatch[g, ω, t] for g in eachindex(data.p_max)) +
             data.unserved_penalty * unserved[ω, t])
    end

    power_investment_cost = sum(data.p_cap_cost[g] * power_capacity[g] for g in eachindex(power_capacity))
    energy_investment_cost = sum(data.e_cap_cost[g] * energy_capacity[i]
        for (i, g) in enumerate(data.energy_indices))

    return (
        termination_status = termination_status(model),
        objective = objective_value(model),
        power_investment_cost = power_investment_cost,
        energy_investment_cost = energy_investment_cost,
        annual_expected_operating_cost = sum(scenario_operating_cost),
        power_capacity_mw = power_capacity,
        energy_indices = copy(data.energy_indices),
        energy_capacity_mwh = energy_capacity,
        dispatch_mw = dispatch,
        load_shift_fraction = value.(variables.sigma),
        unserved_mw = unserved,
        state_of_charge_mwh = energy_state,
        power_balance_shadow_price = balance_dual,
        scenario_operating_cost = scenario_operating_cost,
    )
end

solve_model (generic function with 1 method)

In [34]:
function display_diagnostics(data, results; max_scenarios = 3)
    max_scenarios >= 1 || throw(ArgumentError("max_scenarios must be at least 1"))
    nω = length(data.duration)
    resource_types = unique(string.(data.resource_type))
    html_escape(value) = replace(string(value),
        "&" => "&amp;", "<" => "&lt;", ">" => "&gt;", "\"" => "&quot;", "'" => "&#39;")
    function html_table(headers, rows)
        header_html = join(("<th>$(html_escape(header))</th>" for header in headers))
        rows_html = join(("<tr>" * join(("<td>$(html_escape(value))</td>" for value in row)) * "</tr>"
            for row in rows))
        return HTML("<table style='border-collapse:collapse'><thead><tr>" * header_html *
            "</tr></thead><tbody>" * rows_html * "</tbody></table>")
    end

    println("Termination status: ", results.termination_status)
    println(@sprintf("Total objective cost: %.2f", results.objective))
    println(@sprintf("  Power capacity investment: %.2f", results.power_investment_cost))
    println(@sprintf("  Energy capacity investment: %.2f", results.energy_investment_cost))
    println(@sprintf("  Annual expected operating cost: %.2f", results.annual_expected_operating_cost))

    capacity_rows = Vector{Vector{String}}()
    for resource_type in resource_types
        p_mw = sum(results.power_capacity_mw[g]
            for g in eachindex(data.p_max) if string(data.resource_type[g]) == resource_type)
        e_mwh = sum((results.energy_capacity_mwh[i]
            for (i, g) in enumerate(data.energy_indices)
            if string(data.resource_type[g]) == resource_type); init = 0.0)
        push!(capacity_rows, [resource_type, @sprintf("%.3f", p_mw), @sprintf("%.3f", e_mwh)])
    end
    println("\nBuilt capacity by resource type:")
    display(html_table(["Resource type", "Power (MW)", "Energy (MWh)"], capacity_rows))

    if nω <= max_scenarios
        selected = collect(1:nω)
    else
        candidates = [argmax(results.scenario_operating_cost), argmax(data.duration), argmax(data.kappa)]
        selected = unique(candidates)[1:min(max_scenarios, length(unique(candidates)))]
    end
    println("\nShowing hourly dispatch for scenario(s): ", join(selected, ", "),
        " (", length(selected), " of ", nω, "). For larger sets, selected by highest annual scenario cost, longest duration, and highest curtailment depth.")

    for ω in selected
        println("\nScenario $ω — start hour $(data.start_hour[ω]), duration $(data.duration[ω]) h, depth $(data.kappa[ω]), weight $(data.scenario_weight[ω])")
        headers = ["Hour", "Demand MW", "Load shift %", "Curtailment MW"]
        append!(headers, ["$resource_type MW" for resource_type in resource_types])
        append!(headers, ["Unserved MW", "Balance dual"])
        rows = Vector{Vector{String}}()
        for t in 1:data.duration[ω]
            hour = mod(data.start_hour[ω] + t - 1, 24)
            curtailed_mw = (data.kappa[ω] - results.load_shift_fraction[ω, t]) * data.demand[ω, t]
            cells = [string(hour), @sprintf("%.3f", data.demand[ω, t]),
                @sprintf("%.1f%%", 100 * results.load_shift_fraction[ω, t]), @sprintf("%.3f", curtailed_mw)]
            for resource_type in resource_types
                mw = sum(results.dispatch_mw[g, ω, t]
                    for g in eachindex(data.p_max) if string(data.resource_type[g]) == resource_type)
                push!(cells, @sprintf("%.3f", mw))
            end
            append!(cells, [@sprintf("%.3f", results.unserved_mw[ω, t]),
                @sprintf("%.3f", results.power_balance_shadow_price[ω, t])])
            push!(rows, cells)
        end
        display(html_table(headers, rows))
    end

    println("\nBalance duals are in currency/MW for the equality `dispatch + unserved = remaining curtailed load`; they include scenario-weight and annual event-count scaling. They are marginal objective changes for a one-unit increase in the right-hand side, holding the other model data fixed.")
    return nothing
end

display_diagnostics (generic function with 1 method)

## Synthetic smoke test

This small example uses two resource candidates—one BESS and one VPP—and one event. The shift decision is optimized. Checks verify event balance, load-shift limits, and energy-state bounds. The diagnostic display shows the sole scenario; larger scenario sets are summarized with up to three representative hourly tables.

In [41]:
data = (
    p_max = [4.0, 4.0],
    p_cap_cost = [40.0, 80.0],
    resource_type = ["BESS", "VPP"],
    energy_indices = [1],
    e_max = [1.0, 0],
    e_cap_cost = [10.0, 0.0],
    eta = [0.9, 1.0],
    marginal_cost = [5.0, 20.0],
    emissions_rate = [0.0, 0.5],
    availability = reshape([1.0, 1.0, 0.8, 0.8], 1, 2, 2),
    scenario_weight = [1.0],
    duration = [2],
    kappa = [0.5],
    start_hour = [16],
    demand = [8.0 8.0],
    sigma_max = 0.02,
    events_per_year = 19.0,
    carbon_price = 50.0,
    unserved_penalty = 1000.0,
)

(p_max = [4.0, 4.0], p_cap_cost = [40.0, 80.0], resource_type = ["BESS", "VPP"], energy_indices = [1], e_max = [1.0, 0.0], e_cap_cost = [10.0, 0.0], eta = [0.9, 1.0], marginal_cost = [5.0, 20.0], emissions_rate = [0.0, 0.5], availability = [1.0 1.0;;; 0.8 0.8], scenario_weight = [1.0], duration = [2], kappa = [0.5], start_hour = [16], demand = [8.0 8.0], sigma_max = 0.02, events_per_year = 19.0, carbon_price = 50.0, unserved_penalty = 1000.0)

In [42]:
results = solve_model(data)
display_diagnostics(data, results)

for ω in eachindex(data.duration), t in 1:data.duration[ω]
    required_reduction = (data.kappa[ω] - results.load_shift_fraction[ω, t]) * data.demand[ω, t]
    delivered = sum(results.dispatch_mw[g, ω, t] for g in eachindex(data.p_max)) +
                results.unserved_mw[ω, t]
    @assert isapprox(delivered, required_reduction; atol = 1e-7)
end
for (i, g) in enumerate(data.energy_indices), ω in eachindex(data.duration),
    t in 1:(data.duration[ω] + 1)
    @assert -1e-7 <= results.state_of_charge_mwh[i, ω, t] <= results.energy_capacity_mwh[i] + 1e-7
end
@assert all(-1e-7 <= s <= data.sigma_max + 1e-7 for s in results.load_shift_fraction)
@assert all(isfinite(results.power_balance_shadow_price[ω, t])
    for ω in eachindex(data.duration) for t in 1:data.duration[ω])
@assert isapprox(results.objective,
    results.power_investment_cost + results.energy_investment_cost +
    results.annual_expected_operating_cost; atol = 1e-6)
println("Smoke-test balance, shadow-price, cost, load-shift, and energy-state checks passed.")

Resource type,Power (MW),Energy (MWh)
BESS,1.033,1.000
VPP,3.767,0.000


Hour,Demand MW,Load shift %,Curtailment MW,BESS MW,VPP MW,Unserved MW,Balance dual
16,8.000,2.0%,3.840,0.073,3.767,0.000,877.222
17,8.000,2.0%,3.840,0.827,3.013,0.000,927.222


Set parameter Username
Academic license - for non-commercial use only - expires 2027-09-11
Termination status: OPTIMAL
Total objective cost: 6235.07
  Power capacity investment: 342.67
  Energy capacity investment: 10.00
  Annual expected operating cost: 5882.40

Built capacity by resource type:

Showing hourly dispatch for scenario(s): 1 (1 of 1). For larger sets, selected by highest annual scenario cost, longest duration, and highest curtailment depth.

Scenario 1 — start hour 16, duration 2 h, depth 0.5, weight 1.0

Balance duals are in currency/MW for the equality `dispatch + unserved = remaining curtailed load`; they include scenario-weight and annual event-count scaling. They are marginal objective changes for a one-unit increase in the right-hand side, holding the other model data fixed.
Smoke-test balance, shadow-price, cost, load-shift, and energy-state checks passed.


In [57]:
# include("inputs/20261110_212335/model_inputs.jl")# 5k carbon price
include("inputs/20261110_214919/model_inputs.jl") 
# include("inputs/20261110_211245/model_inputs.jl") # base case
results = solve_model(data)
display_diagnostics(data, results)

Resource type,Power (MW),Energy (MWh)
Diesel,15.000,135.000
BESS,0.000,0.000
LDES,0.000,0.000
PV,0.000,0.000
VPP,0.000,0.000


Hour,Demand MW,Load shift %,Curtailment MW,Diesel MW,BESS MW,LDES MW,PV MW,VPP MW,Unserved MW,Balance dual
17,100.000,10.0%,15.000,15.000,0.000,0.000,0.000,0.000,0.000,263.286
18,100.000,10.0%,15.000,15.000,0.000,0.000,0.000,0.000,0.000,263.286
19,100.000,10.0%,15.000,15.000,0.000,0.000,0.000,0.000,0.000,263.286
20,100.000,10.0%,15.000,15.000,0.000,0.000,0.000,0.000,0.000,263.286
21,100.000,10.0%,15.000,15.000,0.000,0.000,0.000,0.000,0.000,263.286
22,100.000,10.0%,15.000,15.000,0.000,0.000,0.000,0.000,0.000,263.286
23,100.000,10.0%,15.000,15.000,0.000,0.000,0.000,0.000,0.000,263.286
0,100.000,10.0%,15.000,15.000,0.000,0.000,0.000,0.000,0.000,263.286
1,100.000,10.0%,15.000,15.000,0.000,0.000,0.000,0.000,0.000,263.286


Hour,Demand MW,Load shift %,Curtailment MW,Diesel MW,BESS MW,LDES MW,PV MW,VPP MW,Unserved MW,Balance dual
16,100.000,10.0%,15.000,15.000,0.000,0.000,0.000,0.000,0.000,313.080
17,100.000,10.0%,15.000,15.000,0.000,0.000,0.000,0.000,0.000,313.080
18,100.000,10.0%,15.000,15.000,0.000,0.000,0.000,0.000,0.000,313.080
19,100.000,10.0%,15.000,15.000,0.000,0.000,0.000,0.000,0.000,313.080
20,100.000,10.0%,15.000,15.000,0.000,0.000,0.000,0.000,0.000,313.080
21,100.000,10.0%,15.000,15.000,0.000,0.000,0.000,0.000,0.000,313.080
22,100.000,10.0%,15.000,15.000,0.000,0.000,0.000,0.000,0.000,313.080
23,100.000,10.0%,15.000,15.000,0.000,0.000,0.000,0.000,0.000,313.080
0,100.000,10.0%,15.000,15.000,0.000,0.000,0.000,0.000,0.000,313.080


Hour,Demand MW,Load shift %,Curtailment MW,Diesel MW,BESS MW,LDES MW,PV MW,VPP MW,Unserved MW,Balance dual
16,100.000,10.0%,15.000,15.000,0.000,0.000,0.000,0.000,0.000,263.286
17,100.000,10.0%,15.000,15.000,0.000,0.000,0.000,0.000,0.000,263.286
18,100.000,10.0%,15.000,15.000,0.000,0.000,0.000,0.000,0.000,263.286


Set parameter Username
Academic license - for non-commercial use only - expires 2027-09-11
Termination status: OPTIMAL
Total objective cost: 1782948.50
  Power capacity investment: 1278616.24
  Energy capacity investment: 6722.26
  Annual expected operating cost: 497610.00

Built capacity by resource type:

Showing hourly dispatch for scenario(s): 14, 7, 1 (3 of 21). For larger sets, selected by highest annual scenario cost, longest duration, and highest curtailment depth.

Scenario 14 — start hour 17, duration 9 h, depth 0.25, weight 0.047619047619047616

Scenario 7 — start hour 16, duration 9 h, depth 0.25, weight 0.047619047619047616

Scenario 1 — start hour 16, duration 3 h, depth 0.25, weight 0.047619047619047616

Balance duals are in currency/MW for the equality `dispatch + unserved = remaining curtailed load`; they include scenario-weight and annual event-count scaling. They are marginal objective changes for a one-unit increase in the right-hand side, holding the other model da